# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

# Print key dataset metadata
print(f"Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Identifier: {metadata['identifier']}")
print(f"Published: {metadata['datePublished']}")
print(f"License: {metadata['license']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant dataset organizes data into `recordSet` entities. Each `recordSet` contains fields, which map to columns in the underlying data files. We'll list the available record sets, fields, and columns using their `@id` attributes.

Note: All entity references use their unique `@id` as per Croissant schema standard.

In [ ]:
# Get list of record sets by @id
record_sets = dataset.metadata.to_json().get('recordSet', [])

if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Record Sets:")
    for rs in record_sets:
        print(f" - @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")

    # For each record set, list available fields and columns
    print("\nFields and Columns for each record set:")
    for rs in record_sets:
        rs_id = rs['@id']
        fields = rs.get('field', [])
        print(f"RecordSet @id: {rs_id}")
        if fields:
            for field in fields:
                f_repr = field['@id'] if isinstance(field, dict) else field
                print(f"  - Field @id: {f_repr}")
        else:
            print("  No fields listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Let's extract records for each record set using their `@id`.

In [ ]:
# Automatically collect record set @ids
record_sets_json = dataset.metadata.to_json().get('recordSet', [])
record_sets_ids = [rs['@id'] for rs in record_sets_json]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field for filtering and normalization. All references use their `@id`.

For demonstration, let's assume the survey record set has the following field @ids:
 - RecordSet @id: `http://mlcommons.org/croissant/RecordSet/MentalHealthSurvey`
 - Numeric Field @id: `http://mlcommons.org/croissant/Field/gad7_score`
 - Group Field @id: `http://mlcommons.org/croissant/Field/village_name` (categorical grouping)

We will filter out records with GAD-7 score > 10, normalize this field, and group results by village.

In [ ]:
# Specify the relevant @ids
record_set_id = 'http://mlcommons.org/croissant/RecordSet/MentalHealthSurvey'
numeric_field_id = 'http://mlcommons.org/croissant/Field/gad7_score'
group_field_id = 'http://mlcommons.org/croissant/Field/village_name'

# Make sure we have loaded DataFrame for this RecordSet
df = dataframes.get(record_set_id)
if df is not None:
    # Filtering
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by categorical field
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print(f"RecordSet @id {record_set_id} not loaded or no DataFrame.")

## 5. Visualization
Visualize distributions of GAD-7 scores and village grouping. All references by `@id`.

We'll plot the distribution of GAD-7 scores and the average GAD-7 score per village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None:
    # Plot distribution of GAD-7 scores
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of GAD-7 Scores ({numeric_field_id})")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

    if group_field_id in df.columns:
        # Plot average GAD-7 scores by village
        village_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        plt.figure(figsize=(10, 5))
        sns.barplot(x=village_means.values, y=village_means.index)
        plt.title(f"Mean GAD-7 Score by Village ({group_field_id})")
        plt.xlabel("Mean GAD-7 Score")
        plt.ylabel("Village Name")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains rich survey data on mental health indicators in Kilifi County, Kenya, referenced fully by Croissant `@id`.
- Using `mlcroissant`, we loaded metadata and records, listing available record sets and their fields.
- GAD-7 scores were analyzed, filtered, normalized, and grouped by village.
- Visualizations reveal both the distribution and spatial clustering of mental health survey scores.

This notebook demonstrates best practices for FAIR and AI-ready data exploration using `mlcroissant` and Croissant schemas.